# POA GTFS → GeoJSON Transformation

Transforms Porto Alegre (EPTC) GTFS data into GeoJSON for map visualization.

**Input:** `arquivo-gtfs/` (stops, shapes, routes, trips, stop_times)  
**Output:** GeoJSON FeatureCollections for stops (points) and route shapes (linestrings)

In [ ]:
**Requires:** `pandas` — run `pip install pandas` if needed.

In [1]:
import pandas as pd
from pathlib import Path
import json

GTFS_DIR = Path("arquivo-gtfs")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. Stops → Point GeoJSON

Each stop becomes a Point feature with `stop_id`, `stop_name`, and optional `stop_code`.

In [2]:
stops = pd.read_csv(GTFS_DIR / "stops.txt")
# Drop rows with missing coordinates
stops = stops.dropna(subset=["stop_lat", "stop_lon"])

features = []
for _, row in stops.iterrows():
    features.append({
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [float(row["stop_lon"]), float(row["stop_lat"])]
        },
        "properties": {
            "stop_id": str(row["stop_id"]),
            "stop_name": str(row["stop_name"]) if pd.notna(row["stop_name"]) else "",
            "stop_code": str(row["stop_code"]) if pd.notna(row["stop_code"]) else None
        }
    })

stops_geojson = {"type": "FeatureCollection", "features": features}
with open(OUTPUT_DIR / "stops.geojson", "w", encoding="utf-8") as f:
    json.dump(stops_geojson, f, ensure_ascii=False, indent=2)

print(f"Wrote {len(features)} stops to output/stops.geojson")

Wrote 5909 stops to output/stops.geojson


## 2. Shapes → LineString GeoJSON

Each shape (route path) becomes a LineString. Points are ordered by `shape_pt_sequence`.

In [3]:
shapes = pd.read_csv(GTFS_DIR / "shapes.txt")
shapes = shapes.sort_values(["shape_id", "shape_pt_sequence"])

features = []
for shape_id, group in shapes.groupby("shape_id"):
    coords = [[float(r["shape_pt_lon"]), float(r["shape_pt_lat"])] for _, r in group.iterrows()]
    features.append({
        "type": "Feature",
        "geometry": {"type": "LineString", "coordinates": coords},
        "properties": {"shape_id": str(shape_id)}
    })

shapes_geojson = {"type": "FeatureCollection", "features": features}
with open(OUTPUT_DIR / "shapes.geojson", "w", encoding="utf-8") as f:
    json.dump(shapes_geojson, f, ensure_ascii=False, indent=2)

print(f"Wrote {len(features)} shapes to output/shapes.geojson")

Wrote 762 shapes to output/shapes.geojson


## 3. Shapes + Routes → LineStrings with route metadata

Joins shapes with trips and routes so each LineString has route name, color, and type.

In [ ]:
routes = pd.read_csv(GTFS_DIR / "routes.txt")
trips = pd.read_csv(GTFS_DIR / "trips.txt")

# One trip per shape (avoid duplicate shapes)
shape_to_route = trips.drop_duplicates("shape_id")[["shape_id", "route_id"]].merge(
    routes[["route_id", "route_short_name", "route_long_name", "route_type", "route_color"]],
    on="route_id",
    how="left"
)

features = []
for shape_id, group in shapes.groupby("shape_id"):
    coords = [[float(r["shape_pt_lon"]), float(r["shape_pt_lat"])] for _, r in group.iterrows()]
    meta = shape_to_route[shape_to_route["shape_id"] == shape_id]
    props = {"shape_id": str(shape_id)}
    if len(meta) > 0:
        r = meta.iloc[0]
        props.update({
            "route_id": str(r["route_id"]),
            "route_short_name": str(r["route_short_name"]) if pd.notna(r["route_short_name"]) else "",
            "route_long_name": str(r["route_long_name"]) if pd.notna(r["route_long_name"]) else "",
            "route_type": int(r["route_type"]) if pd.notna(r["route_type"]) else None,
            "route_color": f"#{r['route_color']}" if pd.notna(r["route_color"]) and str(r["route_color"]) != "nan" else None
        })
    features.append({
        "type": "Feature",
        "geometry": {"type": "LineString", "coordinates": coords},
        "properties": props
    })

routes_geojson = {"type": "FeatureCollection", "features": features}
with open(OUTPUT_DIR / "routes.geojson", "w", encoding="utf-8") as f:
    json.dump(routes_geojson, f, ensure_ascii=False, indent=2)

print(f"Wrote {len(features)} route shapes to output/routes.geojson")

## Summary

| Output | Type | Description |
|--------|------|-------------|
| `output/stops.geojson` | Points | All transit stops |
| `output/shapes.geojson` | LineStrings | Route paths (shape_id only) |
| `output/routes.geojson` | LineStrings | Route paths with route metadata (name, color, type) |